# Stage 10 Explainer Notebook

This notebook is a study-first tracing version of the stage scripts.
Use it to run scripts step-by-step, inspect middle artifacts, and verify outputs.

## Prerequisites
- Run this notebook from `red-book/src/stage-10` if possible.
- Install dependencies first (for the stage):
  - `pip install -r requirements.txt`
  - optional: `pip install -r requirements-optional.txt`
- Notebook execution policy: run cells in order.

In [ ]:
from __future__ import annotations

import csv
import json
import subprocess
import sys
from pathlib import Path

STAGE = 10
CWD = Path.cwd()

if (CWD / f"run_all_stage{STAGE}.ps1").exists():
    STAGE_DIR = CWD
else:
    STAGE_DIR = Path(r"C:/Users/luixj/AI/red-book/src/stage-10").resolve()

RESULTS_DIR = STAGE_DIR / "results"

print("Stage:", STAGE)
print("Stage dir:", STAGE_DIR)
print("Results dir:", RESULTS_DIR)


def run_cmd(cmd, cwd=STAGE_DIR):
    if isinstance(cmd, str):
        cmd_list = cmd.split()
    else:
        cmd_list = cmd
    print("$", " ".join(map(str, cmd_list)))
    proc = subprocess.run(
        cmd_list,
        cwd=str(cwd),
        text=True,
        capture_output=True,
    )
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print("[stderr]
" + proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed (exit={proc.returncode}): {cmd_list}")
    return proc


def snapshot_results():
    if not RESULTS_DIR.exists():
        return {}
    return {p.name: p.stat().st_mtime for p in RESULTS_DIR.glob('*') if p.is_file()}


def diff_results(before, after):
    new_files = sorted([name for name in after if name not in before])
    changed_files = sorted([
        name for name in after
        if name in before and after[name] != before[name]
    ])
    return new_files, changed_files


def run_script(script_name: str):
    before = snapshot_results()
    script_path = STAGE_DIR / script_name
    if not script_path.exists():
        raise FileNotFoundError(f"Missing script: {script_path}")
    run_cmd([sys.executable, script_name])
    after = snapshot_results()
    new_files, changed_files = diff_results(before, after)
    print("new files:", new_files)
    print("changed files:", changed_files)


def list_results(limit=50):
    if not RESULTS_DIR.exists():
        print("results directory does not exist yet")
        return
    files = sorted([p for p in RESULTS_DIR.glob('*') if p.is_file()])
    print(f"total files: {len(files)}")
    for p in files[:limit]:
        print(f"- {p.name} ({p.stat().st_size} bytes)")


def preview_csv(filename: str, rows: int = 5):
    path = RESULTS_DIR / filename
    if not path.exists():
        print("missing:", path)
        return
    with path.open('r', encoding='utf-8', newline='') as f:
        reader = csv.reader(f)
        for i, row in enumerate(reader):
            print(row)
            if i + 1 >= rows:
                break


def preview_json(filename: str, rows: int = 3):
    path = RESULTS_DIR / filename
    if not path.exists():
        print("missing:", path)
        return
    if filename.endswith('.jsonl'):
        with path.open('r', encoding='utf-8') as f:
            for i, line in enumerate(f):
                print(line.rstrip())
                if i + 1 >= rows:
                    break
    else:
        data = json.loads(path.read_text(encoding='utf-8'))
        if isinstance(data, list):
            print(data[:rows])
        elif isinstance(data, dict):
            keys = list(data.keys())[:rows]
            print({k: data[k] for k in keys})
        else:
            print(data)


## 1) Preflight Checks

In [ ]:
run_cmd([sys.executable, '--version'])

runner = STAGE_DIR / 'run_all_stage10.ps1'
verifier = STAGE_DIR / 'verify_stage10_outputs.ps1'
print('runner exists:', runner.exists(), runner)
print('verifier exists:', verifier.exists(), verifier)
print('lab scripts:', sorted(p.name for p in STAGE_DIR.glob('lab*.py')))


## 2) Run Intermediate Topic Scripts (Trace Middle Artifacts)

In [ ]:
topic_scripts = [
  "topic00_integration_flow_intermediate.py",
  "topic01_data_contracts_intermediate.py",
  "topic02_feature_pipeline_intermediate.py",
  "topic03_ml_prediction_intermediate.py",
  "topic04_retrieval_context_intermediate.py",
  "topic05_llm_reasoning_intermediate.py",
  "topic06_api_serving_intermediate.py",
  "topic07_evaluation_regression_intermediate.py",
  "topic08_ops_release_intermediate.py"
]
print('topic scripts:', topic_scripts)
for script in topic_scripts:
    print('
=== running', script, '===')
    run_script(script)


## 3) Run Stage Labs (Full Workflow Artifacts)

In [ ]:
lab_scripts = [
  "lab01_end_to_end_baseline.py",
  "lab02_pipeline_contract_validation.py",
  "lab03_incident_diagnosis_and_fix.py",
  "lab04_baseline_to_production_integration.py"
]
print('lab scripts:', lab_scripts)
for script in lab_scripts:
    print('
=== running', script, '===')
    run_script(script)


## 4) Optional: Run Stage Fail-Fast Runner

In [ ]:
run_cmd([
    'powershell', '-ExecutionPolicy', 'Bypass',
    '-File', f'run_all_stage{STAGE}.ps1'
], cwd=STAGE_DIR)


## 5) Verify Required Outputs

In [ ]:
verify_script = STAGE_DIR / f'verify_stage{STAGE}_outputs.ps1'
if verify_script.exists():
    run_cmd([
        'powershell', '-ExecutionPolicy', 'Bypass',
        '-File', verify_script.name
    ], cwd=STAGE_DIR)
else:
    print('verify script not found:', verify_script)


## 6) Inspect Results Quickly

In [ ]:
list_results()

candidate_csv = [
    'before_after_metrics.csv',
    'lab1_metrics_comparison.csv',
    'lab4_metrics_comparison.csv',
]
for name in candidate_csv:
    if (RESULTS_DIR / name).exists():
        print('
preview csv:', name)
        preview_csv(name, rows=6)

candidate_json = [
    'decision_log.md',
    'verification_report.md',
    'reproducibility.md',
    'lab1_baseline_outputs.jsonl',
    'lab1_outputs.jsonl',
]
for name in candidate_json:
    if (RESULTS_DIR / name).exists() and name.endswith('.jsonl'):
        print('
preview jsonl:', name)
        preview_json(name, rows=3)
